# TechOps Intelligence Platform
## Notebook 09 — FastAPI Backend + Streamlit UI

**Company:** FinTechFlow — B2B Payment Processor  
**Goal:** Build REST API and demo interface for the agent pipeline

### What This Notebook Does
1. Build FastAPI backend wrapping the 4-agent pipeline
2. Test all API endpoints
3. Build Streamlit demo UI
4. Run end-to-end demo test

In [5]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

os.environ["LANGCHAIN_TRACING_V2"]          = "false"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"
os.environ["LANGCHAIN_API_KEY"]              = ""
os.environ["ANONYMIZED_TELEMETRY"]           = "False"
os.environ["TOKENIZERS_PARALLELISM"]         = "false"

from pathlib import Path

PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Setup complete")
print(f"Project root: {PROJECT_ROOT}")

Setup complete
Project root: C:\Users\sudha\techops-intelligence


## 1. FastAPI Application
REST API wrapping the full 4-agent pipeline
Endpoints:
- POST /analyse  → run pipeline on incident text
- GET  /health   → system health check
- GET  /metrics  → pipeline usage statistics

In [6]:
fastapi_code = '''"""
TechOps Intelligence Platform — FastAPI Backend
FinTechFlow incident intelligence API

Endpoints:
    POST /analyse  → run full 4-agent pipeline
    GET  /health   → system health status
    GET  /metrics  → usage statistics
"""
import os
import sys
import time
import uuid
from pathlib import Path
from typing import Optional, List
from collections import defaultdict

os.environ["LANGCHAIN_TRACING_V2"]          = "false"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"
os.environ["LANGCHAIN_API_KEY"]              = ""
os.environ["ANONYMIZED_TELEMETRY"]           = "False"
os.environ["TOKENIZERS_PARALLELISM"]         = "false"

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

from src.agents.pipeline import run_incident


# ── App Setup ─────────────────────────────────────────────
app = FastAPI(
    title       = "TechOps Intelligence API",
    description = "FinTechFlow incident intelligence powered by LangGraph",
    version     = "1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],
    allow_credentials = True,
    allow_methods     = ["*"],
    allow_headers     = ["*"]
)


# ── Request / Response Models ──────────────────────────────
class IncidentRequest(BaseModel):
    incident_text: str = Field(
        ...,
        min_length  = 10,
        max_length  = 5000,
        description = "Raw incident description or alert text"
    )
    source: str = Field(
        default     = "api",
        description = "Source of the incident (api/streamlit/pagerduty)"
    )


class IncidentResponse(BaseModel):
    incident_id         : str
    severity            : str
    category            : str
    severity_confidence : float
    category_confidence : float
    severity_source     : str
    root_cause          : str
    diagnosis_confidence: float
    resolution_steps    : List[str]
    estimated_time_mins : int
    escalation_needed   : bool
    stakeholder_message : str
    postmortem_draft    : str
    agent_latencies     : dict
    total_latency_s     : float
    error               : Optional[str]


class HealthResponse(BaseModel):
    status     : str
    version    : str
    pipeline   : str
    llm_model  : str
    collections: dict


class MetricsResponse(BaseModel):
    total_incidents   : int
    avg_latency_s     : float
    severity_breakdown: dict
    category_breakdown: dict


# ── In-memory metrics store ────────────────────────────────
_metrics = {
    "total"         : 0,
    "latencies"     : [],
    "severities"    : defaultdict(int),
    "categories"    : defaultdict(int)
}


# ── Endpoints ─────────────────────────────────────────────
@app.post("/analyse", response_model=IncidentResponse)
async def analyse_incident(request: IncidentRequest):
    """
    Run full 4-agent pipeline on incident text.
    Returns severity, root cause, resolution steps,
    stakeholder message and postmortem draft.
    """
    if not request.incident_text.strip():
        raise HTTPException(
            status_code = 400,
            detail      = "incident_text cannot be empty"
        )

    incident_id = f"FTF-{str(uuid.uuid4())[:8].upper()}"

    try:
        result = run_incident(request.incident_text)

        # Update metrics
        total_lat = result["agent_latencies"].get("total", 0)
        _metrics["total"]                              += 1
        _metrics["latencies"].append(total_lat)
        _metrics["severities"][result["severity"]]    += 1
        _metrics["categories"][result["category"]]    += 1

        return IncidentResponse(
            incident_id          = incident_id,
            severity             = result.get("severity", "P2"),
            category             = result.get("category", "application"),
            severity_confidence  = result.get("severity_confidence", 0.0),
            category_confidence  = result.get("category_confidence", 0.0),
            severity_source      = result.get("severity_source", "distilbert"),
            root_cause           = result.get("root_cause", ""),
            diagnosis_confidence = result.get("diagnosis_confidence", 0.0),
            resolution_steps     = result.get("resolution_steps", []),
            estimated_time_mins  = result.get("estimated_time_mins", 30),
            escalation_needed    = result.get("escalation_needed", False),
            stakeholder_message  = result.get("stakeholder_message", ""),
            postmortem_draft     = result.get("postmortem_draft", ""),
            agent_latencies      = result.get("agent_latencies", {}),
            total_latency_s      = total_lat,
            error                = result.get("error")
        )

    except Exception as e:
        raise HTTPException(
            status_code = 500,
            detail      = f"Pipeline error: {str(e)}"
        )


@app.get("/health", response_model=HealthResponse)
async def health_check():
    """System health check."""
    try:
        import chromadb
        from pathlib import Path
        client = chromadb.PersistentClient(
            path=str(Path(__file__).resolve().parents[1]
                     / "data/embeddings/chroma_db")
        )
        collections = {
            col.name: client.get_collection(col.name).count()
            for col in client.list_collections()
        }
    except Exception:
        collections = {}

    return HealthResponse(
        status      = "healthy",
        version     = "1.0.0",
        pipeline    = "triage → diagnosis → resolution → comms",
        llm_model   = "qwen2.5:7b",
        collections = collections
    )


@app.get("/metrics", response_model=MetricsResponse)
async def get_metrics():
    """Pipeline usage statistics."""
    total    = _metrics["total"]
    lats     = _metrics["latencies"]
    avg_lat  = round(sum(lats) / len(lats), 2) if lats else 0.0

    return MetricsResponse(
        total_incidents    = total,
        avg_latency_s      = avg_lat,
        severity_breakdown = dict(_metrics["severities"]),
        category_breakdown = dict(_metrics["categories"])
    )


@app.get("/")
async def root():
    return {
        "name"       : "TechOps Intelligence API",
        "company"    : "FinTechFlow",
        "version"    : "1.0.0",
        "docs"       : "/docs",
        "endpoints"  : ["/analyse", "/health", "/metrics"]
    }
'''

api_path = PROJECT_ROOT / "src/api/main.py"
api_path.parent.mkdir(parents=True, exist_ok=True)
with open(api_path, 'w', encoding='utf-8') as f:
    f.write(fastapi_code)

print(f"FastAPI app saved: {api_path}")

FastAPI app saved: C:\Users\sudha\techops-intelligence\src\api\main.py


## 2. Test FastAPI Endpoints
Start server and test all endpoints

In [7]:
import subprocess
import time
import requests

# Start FastAPI server in background
print("Starting FastAPI server...")
server = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "src.api.main:app",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--reload"
    ],
    cwd    = str(PROJECT_ROOT),
    stdout = subprocess.PIPE,
    stderr = subprocess.PIPE
)

# Wait for server to start
time.sleep(8)
print("Server started — testing endpoints...\n")

BASE_URL = "http://localhost:8000"

# Test 1 — Root endpoint
print("=== GET / ===")
r = requests.get(f"{BASE_URL}/")
print(f"Status : {r.status_code}")
print(f"Body   : {r.json()}\n")

# Test 2 — Health check
print("=== GET /health ===")
r = requests.get(f"{BASE_URL}/health")
print(f"Status : {r.status_code}")
health = r.json()
print(f"Status : {health['status']}")
print(f"Version: {health['version']}")
print(f"Collections:")
for name, count in health['collections'].items():
    print(f"  {name:20}: {count:,}\n")

# Test 3 — Analyse incident
print("=== POST /analyse ===")
incident = {
    "incident_text": "PostgreSQL connection refused ECONNREFUSED port 5432 payment-service max_connections=200 reached 50000 transactions pending",
    "source"       : "test"
}
r = requests.post(
    f"{BASE_URL}/analyse",
    json    = incident,
    timeout = 300
)
print(f"Status : {r.status_code}")
if r.status_code == 200:
    result = r.json()
    print(f"Incident ID  : {result['incident_id']}")
    print(f"Severity     : {result['severity']} ({result['severity_confidence']:.2f})")
    print(f"Category     : {result['category']} ({result['category_confidence']:.2f})")
    print(f"Root cause   : {result['root_cause'][:120]}")
    print(f"Steps        : {len(result['resolution_steps'])}")
    print(f"Total latency: {result['total_latency_s']:.2f}s")
    print(f"Slack message: {result['stakeholder_message'][:150]}")
else:
    print(f"Error: {r.text}")

# Test 4 — Metrics
print("\n=== GET /metrics ===")
r = requests.get(f"{BASE_URL}/metrics")
print(f"Status : {r.status_code}")
print(f"Body   : {r.json()}")

Starting FastAPI server...
Server started — testing endpoints...

=== GET / ===
Status : 200
Body   : {'name': 'TechOps Intelligence API', 'company': 'FinTechFlow', 'version': '1.0.0', 'docs': '/docs', 'endpoints': ['/analyse', '/health', '/metrics']}

=== GET /health ===
Status : 200
Status : healthy
Version: 1.0.0
Collections:
=== POST /analyse ===
Status : 500
Error: {"detail":"Pipeline error: [Errno 22] Invalid argument"}

=== GET /metrics ===
Status : 200
Body   : {'total_incidents': 0, 'avg_latency_s': 0.0, 'severity_breakdown': {}, 'category_breakdown': {}}


## 3. Streamlit Demo UI
Interactive demo interface for the agent pipeline
Shows live agent progress and final results

In [8]:
streamlit_code = '''"""
TechOps Intelligence Platform — Streamlit Demo UI
FinTechFlow incident intelligence demo

Run: streamlit run src/ui/app.py
"""
import os
import sys
import time
import requests
from pathlib import Path

os.environ["LANGCHAIN_TRACING_V2"]          = "false"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"
os.environ["LANGCHAIN_API_KEY"]              = ""
os.environ["ANONYMIZED_TELEMETRY"]           = "False"
os.environ["TOKENIZERS_PARALLELISM"]         = "false"

import streamlit as st

# ── Page Config ───────────────────────────────────────────
st.set_page_config(
    page_title = "TechOps Intelligence — FinTechFlow",
    page_icon  = "🔧",
    layout     = "wide"
)

API_URL = "http://localhost:8000"

# ── Header ────────────────────────────────────────────────
st.title("TechOps Intelligence Platform")
st.caption("FinTechFlow B2B Payment Processor — Incident Intelligence System")
st.divider()

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("About")
    st.markdown("""
    **Multimodal Agentic RAG Pipeline**

    4-agent LangGraph system for IT incident intelligence:
    - Triage Agent — severity + category
    - Diagnosis Agent — root cause via RAG
    - Resolution Agent — runbook steps
    - Comms Agent — stakeholder message

    **Stack**
    - LLM: Qwen2.5 7B (Ollama)
    - Vector DB: ChromaDB
    - Classifier: DistilBERT
    - Retrieval: BM25 + Semantic + Reranking
    """)

    st.divider()
    st.header("Demo Scenarios")

    demo_scenarios = {
        "Database P1 — PostgreSQL": "ALERT: payment-service pods failing health checks. PostgreSQL connection refused on port 5432. Error: ECONNREFUSED max_connections=200 reached. 50,000 transactions pending. Team: payments-sre",
        "Security P1 — Vault Sealed": "CRITICAL: HashiCorp Vault sealed unexpectedly during AWS KMS key rotation. All microservices unable to read secrets. payment-service, order-service, fraud-detection all failing. Team: security-sre",
        "Kubernetes P1 — CrashLoop": "ERROR: payment-service pod CrashLoopBackOff restarts=24 OOMKilled memory limit 4Gi exceeded. 15 pods evicted from EKS cluster. Transactions queuing. Team: platform-sre",
        "Network P2 — Kafka Lag": "WARNING: Kafka consumer group lag reached 2.4M messages on topic=payment-events. transaction-processor falling behind. ETA to backlog clear unknown. Team: platform-sre",
        "Monitoring P2 — Alert Storm": "WARNING: PagerDuty alert storm. 400 alerts in 5 minutes from payment-service. Alert Manager misconfigured. Real P1 incidents may be masked. Team: platform-sre"
    }

    selected = st.selectbox(
        "Load demo scenario:",
        ["-- Select --"] + list(demo_scenarios.keys())
    )

    if selected != "-- Select --":
        st.session_state["demo_text"] = demo_scenarios[selected]

    st.divider()

    # System health
    st.header("System Status")
    try:
        health = requests.get(f"{API_URL}/health", timeout=5).json()
        st.success(f"API: {health['status']}")
        st.caption(f"LLM: {health['llm_model']}")
        for name, count in health.get("collections", {}).items():
            st.caption(f"{name}: {count:,} docs")
    except Exception:
        st.error("API offline — start FastAPI server")

# ── Main Input ────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("Incident Input")
    incident_text = st.text_area(
        "Paste incident alert or describe the issue:",
        value       = st.session_state.get("demo_text", ""),
        height      = 150,
        placeholder = "e.g. PostgreSQL connection refused on port 5432..."
    )

with col2:
    st.subheader("Quick Info")
    st.info("""
    **What this system does:**

    1. Classifies severity (P1/P2/P3)
    2. Identifies root cause
    3. Retrieves fix steps from runbooks
    4. Drafts stakeholder message
    """)

analyse_btn = st.button(
    "Analyse Incident",
    type = "primary",
    use_container_width = True
)

# ── Pipeline Execution ────────────────────────────────────
if analyse_btn and incident_text.strip():
    st.divider()
    st.subheader("Agent Pipeline Execution")

    # Show agent progress
    progress_cols = st.columns(4)
    agent_names   = ["Triage", "Diagnosis", "Resolution", "Comms"]
    agent_status  = {}

    for i, name in enumerate(agent_names):
        with progress_cols[i]:
            agent_status[name] = st.empty()
            agent_status[name].info(f"**{name}**\\nWaiting...")

    # Call API
    with st.spinner("Running incident through agent pipeline..."):
        start = time.time()

        # Update statuses progressively
        agent_status["Triage"].warning("**Triage**\\nRunning...")

        try:
            response = requests.post(
                f"{API_URL}/analyse",
                json    = {"incident_text": incident_text, "source": "streamlit"},
                timeout = 300
            )

            if response.status_code == 200:
                result = response.json()
                total  = time.time() - start

                # Update agent statuses with latencies
                lats = result.get("agent_latencies", {})
                agent_status["Triage"].success(
                    f"**Triage**\\n"
                    f"Done ({lats.get('triage', 0):.2f}s)"
                )
                agent_status["Diagnosis"].success(
                    f"**Diagnosis**\\n"
                    f"Done ({lats.get('diagnosis', 0):.2f}s)"
                )
                agent_status["Resolution"].success(
                    f"**Resolution**\\n"
                    f"Done ({lats.get('resolution', 0):.2f}s)"
                )
                agent_status["Comms"].success(
                    f"**Comms**\\n"
                    f"Done ({lats.get('comms', 0):.2f}s)"
                )

                st.divider()

                # ── Results ──────────────────────────────
                st.subheader("Results")

                # Severity badge + Category
                r1, r2, r3, r4 = st.columns(4)

                severity    = result["severity"]
                sev_colors  = {
                    "P1": "🔴", "P2": "🟠",
                    "P3": "🟡", "P4": "🟢"
                }
                sev_icon = sev_colors.get(severity, "⚪")

                with r1:
                    st.metric(
                        "Severity",
                        f"{sev_icon} {severity}",
                        f"conf: {result['severity_confidence']:.0%}"
                    )
                with r2:
                    st.metric(
                        "Category",
                        result["category"].title(),
                        f"conf: {result['category_confidence']:.0%}"
                    )
                with r3:
                    st.metric(
                        "ETA",
                        f"{result['estimated_time_mins']} mins",
                        "to resolution"
                    )
                with r4:
                    st.metric(
                        "Total Latency",
                        f"{result['total_latency_s']:.1f}s",
                        "pipeline runtime"
                    )

                st.divider()

                # Root cause
                st.subheader("Root Cause")
                st.error(result["root_cause"])

                # Resolution steps
                st.subheader("Resolution Steps")
                steps = result["resolution_steps"]
                if steps:
                    for i, step in enumerate(steps, 1):
                        st.write(f"**{i}.** {step}")
                else:
                    st.write("No steps generated")

                # Two columns for message and postmortem
                mc1, mc2 = st.columns(2)

                with mc1:
                    st.subheader("Stakeholder Message")
                    st.info(result["stakeholder_message"])
                    if st.button("Copy Message"):
                        st.write("Message copied!")

                with mc2:
                    st.subheader("Postmortem Draft")
                    st.text_area(
                        "Postmortem outline:",
                        value  = result["postmortem_draft"],
                        height = 200
                    )

                # Retrieved sources
                with st.expander("Retrieved Sources"):
                    st.write("**Similar Incidents:**")
                    for i, inc in enumerate(
                        result.get("retrieved_incidents", [])[:3], 1
                    ):
                        st.caption(f"{i}. {inc[:150]}")

                    st.write("**Relevant Postmortems:**")
                    for i, pm in enumerate(
                        result.get("retrieved_postmortems", [])[:3], 1
                    ):
                        st.caption(f"{i}. {pm[:150]}")

                # Latency breakdown
                with st.expander("Agent Latency Breakdown"):
                    for agent, lat in lats.items():
                        bar_len = int(lat / 60 * 20)
                        bar     = "█" * bar_len + "░" * (20 - bar_len)
                        st.code(f"{agent:12} [{bar}] {lat:.2f}s")

                # Incident ID
                st.caption(
                    f"Incident ID: {result['incident_id']} | "
                    f"Source: {result.get('severity_source', 'unknown')}"
                )

            else:
                st.error(f"API error {response.status_code}: {response.text}")

        except requests.exceptions.Timeout:
            st.error("Request timed out — pipeline is running, please wait and retry")
        except requests.exceptions.ConnectionError:
            st.error("Cannot connect to API — ensure FastAPI server is running")
        except Exception as e:
            st.error(f"Error: {str(e)}")

elif analyse_btn and not incident_text.strip():
    st.warning("Please enter an incident description")

# ── Footer ────────────────────────────────────────────────
st.divider()
st.caption(
    "TechOps Intelligence Platform v1.0 | "
    "FinTechFlow SRE | "
    "Built with LangGraph + ChromaDB + DistilBERT"
)
'''

ui_path = PROJECT_ROOT / "src/ui/app.py"
ui_path.parent.mkdir(parents=True, exist_ok=True)
with open(ui_path, 'w', encoding='utf-8') as f:
    f.write(streamlit_code)

print(f"Streamlit app saved: {ui_path}")

Streamlit app saved: C:\Users\sudha\techops-intelligence\src\ui\app.py


In [9]:
print("=" * 55)
print("NOTEBOOK 09 - FASTAPI + STREAMLIT COMPLETE")
print("=" * 55)
print("\nFiles created:")
print(f"  API : src/api/main.py")
print(f"  UI  : src/ui/app.py")

print("\nTo run the full demo:")
print("\n  Terminal 1 — FastAPI backend:")
print("    cd C:/Users/sudha/techops-intelligence")
print("    .\\venv\\Scripts\\activate")
print("    uvicorn src.api.main:app --host 0.0.0.0 --port 8000")

print("\n  Terminal 2 — Streamlit UI:")
print("    cd C:/Users/sudha/techops-intelligence")
print("    .\\venv\\Scripts\\activate")
print("    streamlit run src/ui/app.py")

print("\n  Then open: http://localhost:8501")

print("\nNext: Notebook 10 - Docker + Deployment")
print("=" * 55)

NOTEBOOK 09 - FASTAPI + STREAMLIT COMPLETE

Files created:
  API : src/api/main.py
  UI  : src/ui/app.py

To run the full demo:

  Terminal 1 — FastAPI backend:
    cd C:/Users/sudha/techops-intelligence
    .\venv\Scripts\activate
    uvicorn src.api.main:app --host 0.0.0.0 --port 8000

  Terminal 2 — Streamlit UI:
    cd C:/Users/sudha/techops-intelligence
    .\venv\Scripts\activate
    streamlit run src/ui/app.py

  Then open: http://localhost:8501

Next: Notebook 10 - Docker + Deployment


In [2]:
import requests

r = requests.post(
    "http://localhost:8000/analyse",
    json    = {
        "incident_text": "PostgreSQL connection refused port 5432 payment-service",
        "source"       : "test"
    },
    timeout = 300
)

result = r.json()
print("=== ALL FIELDS ===")
for key, val in result.items():
    if isinstance(val, list):
        print(f"{key}: {val[:2] if val else '[]'}")
    elif isinstance(val, str):
        print(f"{key}: {val}")
    else:
        print(f"{key}: {val}")

=== ALL FIELDS ===
incident_id: FTF-4E1A4A8C
severity: P1
category: database
severity_confidence: 0.75
category_confidence: 0.8939999938011169
severity_source: rule_based
root_cause: The incident was caused by an accidental misconfiguration in the AWS Security Group rules, blocking port 5432 for the payment-service pods.
diagnosis_confidence: 0.8
resolution_steps: ['1. Revert the misconfiguration in the AWS Security Group rules to allow port 5432 for the payment-service pods.', '2. Use `SELECT pg_terminate_backend(12345)` to safely kill any idle PostgreSQL connections as needed.']
estimated_time_mins: 30
escalation_needed: True
stakeholder_message: _MESSAGE: PostgreSQL connection to payment-service on port 5432 refused due to misconfigured AWS Security Group; expect resolution in 30 minutes.
postmortem_draft: Title: Misconfigured AWS Security Group Blocking PostgreSQL Connection
Summary: An accidental misconfiguration in the AWS Security Group rules blocked port 5432 for the payment-se